In [ ]:
import oceanbench

oceanbench.__version__

### Open challenger datasets

> Insert here the code that opens the challenger dataset as `challenger_dataset: xarray.Dataset`

In [ ]:
# Open the full 10-day GLO12 forecasts covered by the January 2024 IBIRYS archive
from datetime import datetime
import xarray
import oceanbench

challenger_dataset: xarray.Dataset = oceanbench.datasets.challenger.glo12(
    [datetime(2024, 1, day) for day in (3, 10, 17)]
)

challenger_dataset


### Evaluation configuration

In [ ]:
region = 'ibi'

### Evaluation of challenger dataset using OceanBench

#### Root Mean Square Deviation (RMSD) of variables compared to IBIRYS reanalysis

In [ ]:
import pandas

from oceanbench.core.dataset_utils import DepthLevel, Variable
from oceanbench.core.regions import subset_dataset_to_region
from oceanbench.core.ocean_mask import ocean_mask
from oceanbench.core.rmsd import rmsd

challenger_dataset = subset_dataset_to_region(challenger_dataset, region)
challenger_dataset = challenger_dataset.sel(depth=[level.value for level in DepthLevel], method="nearest")

from oceanbench.core.weekly_stage import prepare_reference_week_dataset

ibirys_zarr = "https://s3.waw3-1.cloudferro.com/oceanbench-bucket/public/ibirys36_1-12_glorys-grid/202401.zarr"
ibirys = xarray.open_zarr(ibirys_zarr, consolidated=True)
ibirys = ibirys.sel(depth=challenger_dataset.depth.values, method="nearest")

lead_days_count = challenger_dataset.sizes["lead_day_index"]

reference_dataset = xarray.concat(
    [
        prepare_reference_week_dataset(
            ibirys.sel(time=pandas.date_range(first_day, periods=lead_days_count)),
            lead_days_count=lead_days_count,
            operation_name="IBIRYS evaluation dataset",
        )
        for first_day in challenger_dataset.first_day_datetime.values
    ],
    dim="first_day_datetime",
).assign_coords(first_day_datetime=challenger_dataset.first_day_datetime)
reference_dataset

In [ ]:
rmsd(
    challenger_dataset=challenger_dataset,
    reference_dataset=reference_dataset,
    variables=[
        Variable.SEA_SURFACE_HEIGHT_ABOVE_GEOID,
        Variable.SEA_WATER_POTENTIAL_TEMPERATURE,
        Variable.SEA_WATER_SALINITY,
        Variable.NORTHWARD_SEA_WATER_VELOCITY,
        Variable.EASTWARD_SEA_WATER_VELOCITY,
    ],
    ocean_mask=ocean_mask(),
)

#### Ocean mask check

First forecast, first lead, surface temperature. Percentage of OceanBench ocean points without IBIRYS.
Map: 0 = IBIRYS available; 1 = missing; blank = outside the ocean mask.

In [ ]:
temperature = reference_dataset[Variable.SEA_WATER_POTENTIAL_TEMPERATURE.key()].isel(
    first_day_datetime=0, lead_day_index=0, depth=0, drop=True
)
mask = ocean_mask().isel(depth=0, drop=True).sel(
    latitude=temperature.latitude, longitude=temperature.longitude, method="nearest"
).assign_coords(latitude=temperature.latitude, longitude=temperature.longitude)
missing = temperature.isnull().where(mask).compute()
print(f"Ocean points without IBIRYS: {100 * float(missing.mean()):.2f}%")
missing.plot(
    x="longitude", y="latitude", cmap="coolwarm", vmin=0, vmax=1,
    cbar_kwargs={"ticks": [0, 1], "label": "0: available; 1: missing"},
)